In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"  # T4 doesn't support bf16
)
model.eval()

SYSTEM = (
    "You are evaluating answers to reading comprehension questions. "
    "Given a paragraph, a question and a candidate answer, decide whether the "
    "candidate answer is correct using only information in the paragraph. "
    "A question may have more than one correct answer. "
    "Reply with exactly one word: True or False."
)

def user_msg(paragraph, question, answer):
    return (f"Paragraph: {paragraph}\n\n"
            f"Question: {question}\n\n"
            f"Candidate answer: {answer}\n\n"
            "Is the candidate answer correct?")

SHOTS = [
    (
        r"""Animated history of the US. Of course the cartoon is highly oversimplified, and most critics consider it one of the weakest parts of the film. But it makes a valid claim which you ignore entirely: That the strategy to promote "gun rights" for white people and to outlaw gun possession by black people was a way to uphold racism without letting an openly terrorist organization like the KKK flourish. Did the 19th century NRA in the southern states promote gun rights for black people? I highly doubt it. But if they didn't, one of their functions was to continue the racism of the KKK. This is the key message of this part of the animation, which is again being ignored by its critics. Buell shooting in Flint. You write: "Fact: The little boy was the class thug, already suspended from school for stabbing another kid with a pencil, and had fought with Kayla the day before". This characterization of a six-year-old as a pencil-stabbing thug is exactly the kind of hysteria that Moore's film warns against. It is the typical right-wing reaction which looks for simple answers that do not contradict the Republican mindset. The kid was a little bastard, and the parents were involved in drugs -- case closed. But why do people deal with drugs? Because it's so much fun to do so? It is by now well documented that the CIA tolerated crack sales in US cities to fund the operation of South American "contras" It is equally well known that the so-called "war on drugs" begun under the Nixon administration is a failure which has cost hundreds of billions and made America the world leader in prison population (both in relative and absolute numbers)""",
        r"""Does the author claim the animated films message is that the NRA upholds racism?""",
        r"""Yes""",
        True
    ),
    (
        r"""Hotel California My first thought: I was going crazy. Twenty-four hours of silence (vacuum, remember); was I hallucinating noises now? I heard it again. It was a fine bell, reminiscent of ancient stone churches and the towering cathedrals I'd seen in documentaries. And accompanying the bell, I saw a light. Now, there were two things here that made ridiculously small amounts of sense. First, the whole in-a-vacuum why's-there-a-bell thing. Second, I was floating in the dark remnants of my broken ship, and any conceivable light sources were not within view; starlight is a distinctly different color and significantly less bright. These signals were the heralds of my saviors. The first words they said to me meant nothing. I wasn't listening; I didn't care; I was going to live; I was going to keep breathing. Next to those, nothing else mattered. The recycled air tasted sweet in my mouth, and all thoughts that crossed my mind were cheap metaphors about life-giving substances and how breathing was like sex, only better. (I reserved the right to revise this opinion later.) When I was done mentally exclaiming over my impossible rescue, I looked around. The ship, it was odd and old, either so outdated or so heavily modified that I couldn't tell what make it was, and somehow, the crew standing around me fit the same description, a singularly atypical amalgamation of folk. And me, I guess I was one more piece in their puzzle. I was one more scrap to weld onto the rest, one more stranded survivor who was found. I was now one of them.""",
        r"""Who is the "Them" the writer refers to being one of?""",
        r"""He felt he was one more person for who nothing else mattered""",
        False
    )
]

def build_messages(paragraph, question, answer, shots=SHOTS):
    msgs = [{"role": "system", "content": SYSTEM}]
    for p, q, a, label in shots:
        msgs.append({"role": "user", "content": user_msg(p, q, a)})
        msgs.append({"role": "assistant", "content": "True" if label else "False"})
    msgs.append({"role": "user", "content": user_msg(paragraph, question, answer)})
    return msgs

# token ids for the two possible answers
TRUE_ID = tok.encode("True", add_special_tokens=False)
FALSE_ID = tok.encode("False", add_special_tokens=False)
assert len(TRUE_ID) == 1 and len(FALSE_ID) == 1, "check tokenisation"
TRUE_ID, FALSE_ID = TRUE_ID[0], FALSE_ID[0]

@torch.no_grad()
def predict(paragraph, question, answer, shots=SHOTS):
    prompt = tok.apply_chat_template(
        build_messages(paragraph, question, answer, shots),
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]
    return bool(logits[TRUE_ID] > logits[FALSE_ID])

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/MultiRC/'

df = pd.read_json(DATA_PATH + "test_83-fixedIds.json")


def get_paragraph_and_questions(df):
    for row in df.itertuples():
        para_id = row[0]
        paragraph = row[1]['paragraph']
        paragraph_text = paragraph['text']
        paragraph_questions = paragraph['questions']
        for question in paragraph_questions:
            question_text = question['question']
            question_id = question['idx']                                        # ADDED
            for answer in question['answers']:
                answer_text = answer['text']
                is_answer = answer['isAnswer']
                yield para_id, question_id, paragraph_text, question_text, answer_text, is_answer   # CHANGED
import re

def clean_paragraph(text):
    text = re.sub(r"<b>Sent \d+: </b>", "", text)  # remove "Sent N:" markers
    text = text.replace("<br>", " ")                # line breaks -> spaces
    text = re.sub(r"\s+", " ", text).strip()        # tidy whitespace
    return text

questions_answers = get_paragraph_and_questions(df)

output_file = DATA_PATH + "qwen3_4b_2shot_test_preds.csv"                        # ADDED

#run model
with open(output_file, "w", encoding="utf-8") as f:
  f.write("para_id,question_id,label,prediction,correct\n")                     # ADDED
  try:
      while True:
          para_id, question_id, paragraph, question, answer, label = next(questions_answers)   # CHANGED
          paragraph = clean_paragraph(paragraph)
          prediction = predict(paragraph, question, answer)
          if prediction == label:
            correct = True
          else:
            correct = False
          f.write(f"{para_id},{question_id},{label},{prediction},{correct}\n")  # CHANGED
          f.flush()                                                              # ADDED

  except StopIteration:
      print("Reached the end of the generator!")

Mounted at /content/drive
Reached the end of the generator!
